# CliniqAI — breast_histology specialist v2, end-to-end (BreakHis)

Replaces the IDC 50x50-patch dataset (root cause of the gatekeeper's breast_histology rejection
bug, and a low-quality source for the specialist itself) with **BreakHis**
(Spanhol et al. 2016) — 7,909 real histopathology images, 700x460px, RGB, four magnifications
(40X/100X/200X/400X), benign vs malignant with 8 histological subtypes. Public, free for
non-commercial research/education (cite Spanhol et al., IEEE TBME 2016).

**Class semantics change, flag this before deploying:** the old specialist output
`idc_negative`/`idc_positive` (invasive ductal carcinoma presence in a patch) is being replaced
with `benign`/`malignant` (tumor-level diagnosis). This is a different clinical question, not
just a quality upgrade — `backend/app/schemas.py` and any frontend display logic that assumes
IDC-style labels will need updating before this checkpoint goes live. If you want to keep
IDC-style output, that's an argument for pairing BreakHis with a fine-tune rather than fully
retraining. Proceeding with `benign`/`malignant` here since that's the direct fit for BreakHis's
own labels — say if you want the 8-subtype multiclass version instead, it's a one-line change
(see Step 1).

**Kaggle mirror used:** `ambarish/breakhis` (`?select=BreaKHis_v1` — the same mirror used across
most published work on this dataset). Folder structure inside:
`BreaKHis_v1/histology_slides/breast/{benign,malignant}/SOB/<subtype>/<patient_folder>/<mag>X/*.png`.
Filenames follow `SOB_{B|M}_{SUBTYPE}-{patient_id}-{mag}-{seq}.png` — parsed with regex below
rather than trusted to directory naming, since folder naming has varied slightly across mirror
re-uploads.

**Bonus for the gatekeeper bug (not fixed in this notebook, but worth knowing):** BreakHis
naturally contains four magnification levels of the same tissue, i.e. real zoom-scale variety —
exactly what the gatekeeper's old 50x50-patch-only positive class was missing. Once this
specialist is trained and validated, swapping BreakHis in as the gatekeeper's breast_histology
positive-class source (instead of / in addition to the resolution-degradation augmentation) is
probably a better fix for the zoom/crop-scale hypothesis than more augmentation on bad data —
worth trying before a third fine-tune attempt.

Run top to bottom. Needs `kaggle.json` already configured (`~/.kaggle/kaggle.json`) and the
`CliniqAI` conda env active.

## Step 0 — Imports

In [1]:
import os
import re
import sys
import json
import time
import shutil
import random
import subprocess
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split

ImageFile.LOAD_TRUNCATED_IMAGES = True
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())


torch: 2.5.0+cu124 | cuda available: True


## Step 1 — Config

In [2]:
PROJECT_ROOT = Path(r"E:\programming\CliniqAI")
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "breast_histology_v2"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODELS_DIR = PROJECT_ROOT / "models" / "specialists"
BACKEND_MODELS_DIR = PROJECT_ROOT / "backend" / "models" / "specialists"
SANITY_DIR = PROJECT_ROOT / "data" / "raw" / "manual_sanity_check"

KAGGLE_DATASET = "ambarish/breakhis"

# Set to True for the 8-subtype multiclass version instead of binary benign/malignant.
MULTICLASS_SUBTYPES = False

IMG_SIZE = 224
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
EARLY_STOP_PATIENCE = 8
MAX_EPOCHS = 40
SEED = 42
CHECKPOINT_NAME = "breast_histology_specialist_v2"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

DATA_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
BACKEND_MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("device:", device)


device: cuda


## Step 2 — Download BreakHis via Kaggle CLI

Uses the pinned `kaggle==1.6.17` (newer `kagglesdk`-backed versions buffer the whole zip in RAM
and MemoryError on datasets this size — established project gotcha).

In [3]:
# Uncomment the line below once if this env doesn't already have the pinned version
# !pip uninstall kaggle -y && pip install "kaggle==1.6.17"

zip_marker = DATA_DIR / "BreaKHis_v1"
if not zip_marker.exists():
    cmd = [
        "kaggle", "datasets", "download",
        "-d", KAGGLE_DATASET,
        "-p", str(DATA_DIR),
        "--unzip",
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError("Kaggle download failed — check kaggle.json and dataset slug above.")
else:
    print("Already downloaded/extracted, skipping.")


Already downloaded/extracted, skipping.


## Step 3 — Index all images, parse label / subtype / magnification / patient

Parsed primarily from the filename (`SOB_{B|M}_{SUBTYPE}-{patient_id}-{mag}-{seq}.png`), which
is the stable part of the official BreakHis convention — falls back to parent-folder name for
magnification (`40X`/`100X`/`200X`/`400X`) if a filename ever doesn't match, since mirror
re-uploads have occasionally altered folder nesting.

In [4]:
FNAME_RE = re.compile(r"SOB_([BM])_([A-Za-z]+)-(\d+-\d+[A-Za-z]{0,3})-(\d+)-(\d+)\.(png|PNG)$")
MAG_DIR_RE = re.compile(r"^(\d+)X$")

SUBTYPE_MAP = {
    "A": "adenosis", "F": "fibroadenoma", "TA": "tubular_adenoma", "PT": "phyllodes_tumor",
    "DC": "ductal_carcinoma", "LC": "lobular_carcinoma", "MC": "mucinous_carcinoma", "PC": "papillary_carcinoma",
}

records = []  # (path, label, subtype, magnification, patient_id)
# NOTE: Windows filesystems are case-insensitive, so globbing *.png and *.PNG separately
# and concatenating matches every file TWICE there (harmless on case-sensitive filesystems,
# but silently doubles the dataset on Windows). Dedupe by resolved path to be safe either way.
_seen_paths = set()
all_pngs = []
for p in list(DATA_DIR.glob("**/*.png")) + list(DATA_DIR.glob("**/*.PNG")):
    rp = p.resolve()
    if rp not in _seen_paths:
        _seen_paths.add(rp)
        all_pngs.append(p)
print(f"Found {len(all_pngs)} .png files under {DATA_DIR}")

skipped = 0
for p in all_pngs:
    m = FNAME_RE.search(p.name)
    if m:
        label = "benign" if m.group(1) == "B" else "malignant"
        subtype = SUBTYPE_MAP.get(m.group(2).upper(), m.group(2).lower())
        patient_id = m.group(3)
        mag = m.group(4) + "X"
    else:
        # fallback: derive from directory structure
        parts = p.parts
        mag_match = next((MAG_DIR_RE.match(part) for part in parts if MAG_DIR_RE.match(part)), None)
        if mag_match is None or ("benign" not in str(p).lower() and "malignant" not in str(p).lower()):
            skipped += 1
            continue
        mag = mag_match.group(0)
        label = "benign" if "benign" in str(p).lower() else "malignant"
        subtype = "unknown"
        patient_id = p.parent.parent.name  # best-effort

    records.append((str(p), label, subtype, mag, patient_id))

print(f"Indexed {len(records)} images, skipped {skipped} unparseable paths.")
label_counts = Counter(r[1] for r in records)
mag_counts = Counter(r[3] for r in records)
subtype_counts = Counter(r[2] for r in records)
print("By label:", dict(label_counts))
print("By magnification:", dict(mag_counts))
print("By subtype:", dict(subtype_counts))
print("Unique patients:", len(set(r[4] for r in records)))


Found 7909 .png files under E:\programming\CliniqAI\data\raw\breast_histology_v2
Indexed 7909 images, skipped 0 unparseable paths.
By label: {'benign': 2480, 'malignant': 5429}
By magnification: {'100X': 2081, '200X': 2013, '400X': 1820, '40X': 1995}
By subtype: {'adenosis': 444, 'fibroadenoma': 1014, 'phyllodes_tumor': 453, 'tubular_adenoma': 569, 'ductal_carcinoma': 3451, 'lobular_carcinoma': 626, 'mucinous_carcinoma': 792, 'papillary_carcinoma': 560}
Unique patients: 81


## Step 4 — Corrupted-file validation pass (project convention)

In [5]:
bad_paths = set()
for path, *_ in records:
    try:
        with Image.open(path) as im:
            im.load()
    except Exception as e:
        bad_paths.add(path)

if bad_paths:
    print(f"Dropping {len(bad_paths)} corrupted/unreadable files.")
    records = [r for r in records if r[0] not in bad_paths]
else:
    print("No corrupted files found.")
print(f"{len(records)} usable images remain.")


No corrupted files found.
7909 usable images remain.


## Step 5 — Patient-grouped, subtype-stratified train/val/test split

BreakHis has multiple images per patient (across magnifications and within a slide). Splitting
by image instead of by patient lets near-duplicate tissue from the same patient leak across
train/val/test and inflates eval numbers. Splits by **patient**.

Stratified by **subtype**, not just binary label — with only 82 patients total and some
subtypes (`phyllodes_tumor`: 3, `adenosis`: 4, `lobular_carcinoma`: 5) having very few patients,
label-only stratification can leave a whole subtype with zero val or test patients by chance
(this happened in practice: v1 of this split gave `adenosis`, `mucinous_carcinoma`, and
`phyllodes_tumor` zero test coverage). Every subtype gets **at least 1 patient in each of
train/val/test** where it has 3+ patients to draw from — this makes per-subtype eval numbers
actually trustworthy instead of silently missing or single-patient. Image-level split
proportions won't land exactly on 80/10/10 as a result (larger subtypes still dominate by image
count) — that's expected and fine; patient-level coverage is the property being guaranteed here,
not an exact image ratio.

In [6]:
patients_by_subtype = defaultdict(set)
for _, label, subtype, _, patient_id in records:
    patients_by_subtype[subtype].add(patient_id)

train_patients, val_patients, test_patients = set(), set(), set()
for subtype in sorted(patients_by_subtype.keys()):
    # skip patients already assigned via another subtype (the subtype patient-counts summed to
    # 82 against 81 unique patients overall -- one patient's images span two subtype labels)
    patients = sorted(p for p in patients_by_subtype[subtype]
                       if p not in train_patients and p not in val_patients and p not in test_patients)
    random.Random(SEED).shuffle(patients)
    n = len(patients)
    if n == 0:
        continue
    n_test = max(1, round(n * 0.1))
    n_val = max(1, round(n * 0.1))
    n_train = n - n_test - n_val
    if n_train < 1:
        # extremely small subtype: guarantee at least 1 train patient even if it means
        # shrinking val/test to make room
        n_train = 1
        remaining = n - n_train
        n_val = remaining // 2
        n_test = remaining - n_val
    train_patients.update(patients[:n_train])
    val_patients.update(patients[n_train:n_train + n_val])
    test_patients.update(patients[n_train + n_val:])

def split_of(patient_id):
    if patient_id in train_patients:
        return "train"
    if patient_id in val_patients:
        return "val"
    return "test"

split_records = defaultdict(list)
for path, label, subtype, mag, patient_id in records:
    split_records[split_of(patient_id)].append((path, label, subtype, mag, patient_id))

for split_name in ("train", "val", "test"):
    label_c = Counter(r[1] for r in split_records[split_name])
    subtype_c = Counter(r[2] for r in split_records[split_name])
    print(f"{split_name}: {len(split_records[split_name])} images | label={dict(label_c)}")
    print(f"  subtype breakdown: {dict(subtype_c)}")

print("\nPatient coverage per subtype (train / val / test / total):")
for subtype in sorted(patients_by_subtype):
    tp = len(patients_by_subtype[subtype] & train_patients)
    vp = len(patients_by_subtype[subtype] & val_patients)
    tep = len(patients_by_subtype[subtype] & test_patients)
    print(f"  {subtype}: {tp} / {vp} / {tep} / {len(patients_by_subtype[subtype])}")


train: 5894 images | label={'benign': 1637, 'malignant': 4257}
  subtype breakdown: {'adenosis': 262, 'fibroadenoma': 778, 'phyllodes_tumor': 158, 'tubular_adenoma': 439, 'ductal_carcinoma': 2749, 'lobular_carcinoma': 449, 'mucinous_carcinoma': 638, 'papillary_carcinoma': 421}
val: 1072 images | label={'benign': 503, 'malignant': 569}
  subtype breakdown: {'adenosis': 61, 'fibroadenoma': 142, 'phyllodes_tumor': 235, 'tubular_adenoma': 65, 'ductal_carcinoma': 357, 'lobular_carcinoma': 74, 'mucinous_carcinoma': 64, 'papillary_carcinoma': 74}
test: 943 images | label={'benign': 340, 'malignant': 603}
  subtype breakdown: {'adenosis': 121, 'fibroadenoma': 94, 'phyllodes_tumor': 60, 'tubular_adenoma': 65, 'ductal_carcinoma': 345, 'lobular_carcinoma': 103, 'mucinous_carcinoma': 90, 'papillary_carcinoma': 65}

Patient coverage per subtype (train / val / test / total):
  adenosis: 2 / 1 / 1 / 4
  ductal_carcinoma: 30 / 4 / 4 / 38
  fibroadenoma: 8 / 1 / 1 / 10
  lobular_carcinoma: 3 / 1 / 1 / 

In [17]:
papillary_patients = set(r[4] for r in test_records if r[2] == "papillary_carcinoma")
print(f"{len(papillary_patients)} unique patients behind the 65 papillary_carcinoma test images:", papillary_patients)

1 unique patients behind the 65 papillary_carcinoma test images: {'15-190EF'}


## Step 6 — Write the standalone Dataset module

Must live in its own importable `.py` file under `scripts/` — inline notebook-cell Dataset
classes silently fall back to single-process loading under Windows `num_workers>0`
multiprocessing. `ImageFile.LOAD_TRUNCATED_IMAGES` is set at module level here too, since worker
subprocesses re-import this module fresh and won't inherit the notebook's setting.

In [7]:
dataset_module_path = SCRIPTS_DIR / "breast_histology_dataset_v2.py"
dataset_module_code = '''"""
Standalone, importable Dataset for breast_histology specialist v2 training (BreakHis).
Must live in its own module (not a notebook cell) for Windows DataLoader multiprocessing
(num_workers > 0) to work -- worker subprocesses re-import this module fresh, so
ImageFile.LOAD_TRUNCATED_IMAGES must be set here at module level, not just in the notebook.
"""
import random
from pathlib import Path
from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True


class BreastHistologyDatasetV2(Dataset):
    """records: list of (path, label_str, subtype, magnification, patient_id)."""

    def __init__(self, records, label_to_idx, transform=None):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def _load(self, idx):
        path, label_str, _subtype, _mag, _patient = self.records[idx]
        img = Image.open(path).convert("RGB")
        img.load()
        label = self.label_to_idx[label_str]
        return img, label

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
        except (OSError, Image.DecompressionBombError):
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
        if self.transform:
            img = self.transform(img)
        return img, label
'''

dataset_module_path.write_text(dataset_module_code, encoding="utf-8")
print(f"Wrote {dataset_module_path}")

sys.path.insert(0, str(SCRIPTS_DIR))
from breast_histology_dataset_v2 import BreastHistologyDatasetV2


Wrote E:\programming\CliniqAI\scripts\breast_histology_dataset_v2.py


## Step 7 — Labels, transforms, class weights

In [8]:
if MULTICLASS_SUBTYPES:
    class_names = sorted(subtype_counts.keys())
else:
    class_names = ["benign", "malignant"]

label_key = (lambda r: r[2]) if MULTICLASS_SUBTYPES else (lambda r: r[1])
label_to_idx = {c: i for i, c in enumerate(class_names)}
print("class_names:", class_names)

def to_labeled(records_subset):
    return [(path, label_key(r), r[2], r[3], r[4]) for r, path in ((r, r[0]) for r in records_subset)]

train_records = to_labeled(split_records["train"])
val_records = to_labeled(split_records["val"])
test_records = to_labeled(split_records["test"])

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_label_counts = Counter(r[1] for r in train_records)
class_counts = np.array([train_label_counts.get(c, 0) for c in class_names], dtype=np.float64)
class_weights = (class_counts.sum() / (len(class_names) * class_counts)).astype(np.float32)
print("train class counts:", dict(zip(class_names, class_counts.astype(int))))
print("class weights (for loss):", dict(zip(class_names, class_weights)))


class_names: ['benign', 'malignant']
train class counts: {'benign': 1637, 'malignant': 4257}
class weights (for loss): {'benign': 1.8002443, 'malignant': 0.69227153}


## Step 8 — Datasets & DataLoaders (weighted sampler for imbalance)

In [9]:
train_ds = BreastHistologyDatasetV2(train_records, label_to_idx, transform=train_transform)
val_ds = BreastHistologyDatasetV2(val_records, label_to_idx, transform=eval_transform)
test_ds = BreastHistologyDatasetV2(test_records, label_to_idx, transform=eval_transform)

sample_weights = np.array([class_weights[label_to_idx[r[1]]] for r in train_records], dtype=np.float32)
sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")


train=5894 val=1072 test=943


## Step 9 — Model

Simple head (`Dropout -> Linear(1280, num_classes)`) to match the head type
`model_loader.py`'s `_build_matching_head()` already expects for this checkpoint slot (same
group as pneumonia/skin_lesion/bone_fracture) — no loader changes needed.

In [10]:
def build_model(num_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, num_classes),
    )
    return model

model = build_model(len(class_names)).to(device, memory_format=torch.channels_last)
print(model.classifier)


Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


## Step 10 — Loss, optimizer, scheduler, AMP

In [11]:
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, device=device))
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)


## Step 11 — Training loop

Per-epoch checkpointing to `.checkpoint_inprogress.pt` on val_loss improvement, with a resume
cell below (Step 12) — reliability/resumability over throughput, per project convention.

In [12]:
checkpoint_inprogress_path = MODELS_DIR / f"{CHECKPOINT_NAME}.checkpoint_inprogress.pt"
history = []
best_val_loss = float("inf")
epochs_no_improve = 0
start_epoch = 0

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

for epoch in range(start_epoch, MAX_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    print(f"epoch {epoch+1:02d} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.4f} | {elapsed:.1f}s")

    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "classes": class_names,
            "num_classes": len(class_names),
            "architecture": "efficientnet_b0",
            "epoch": epoch + 1,
            "best_val_loss": best_val_loss,
        }, checkpoint_inprogress_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs).")
            break

print("Training done. Best val_loss:", best_val_loss)


epoch 01 | train_loss 0.1933 acc 0.8974 | val_loss 0.3426 acc 0.8937 | 114.0s
epoch 02 | train_loss 0.0978 acc 0.9561 | val_loss 0.5162 acc 0.8722 | 76.5s
epoch 03 | train_loss 0.0786 acc 0.9635 | val_loss 1.1389 acc 0.8377 | 76.9s
epoch 04 | train_loss 0.0572 acc 0.9742 | val_loss 0.6565 acc 0.8172 | 76.1s
epoch 05 | train_loss 0.0523 acc 0.9774 | val_loss 0.6031 acc 0.8750 | 76.2s
epoch 06 | train_loss 0.0279 acc 0.9885 | val_loss 0.9536 acc 0.8377 | 76.5s
epoch 07 | train_loss 0.0460 acc 0.9800 | val_loss 1.0546 acc 0.8405 | 76.5s
epoch 08 | train_loss 0.0483 acc 0.9812 | val_loss 0.5603 acc 0.8675 | 76.3s
epoch 09 | train_loss 0.0263 acc 0.9891 | val_loss 0.8390 acc 0.8517 | 76.3s
Early stopping at epoch 9 (no improvement for 8 epochs).
Training done. Best val_loss: 0.342648074684192


## Step 12 — Resume from checkpoint (only run this if the kernel was restarted mid-training)

Restarting the kernel wipes all in-memory state — re-run Steps 0-10 first (imports through
model/optimizer setup), then run this cell instead of starting Step 11 from scratch.

In [13]:
# resume_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
# model.load_state_dict(resume_ckpt["model_state_dict"])
# start_epoch = resume_ckpt["epoch"]
# best_val_loss = resume_ckpt["best_val_loss"]
# print(f"Resumed from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")
# # then re-run the Step 11 training loop cell, which will pick up from start_epoch


## Step 13 — Final save (project root + backend copy)

In [14]:
best_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

final_path = MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
torch.save(best_ckpt, final_path)

backend_final_path = BACKEND_MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
shutil.copy2(final_path, backend_final_path)

history_path = MODELS_DIR / f"{CHECKPOINT_NAME}.history.json"
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

print("Saved:", final_path)
print("Saved:", backend_final_path)
print("Saved:", history_path)


C:\Users\anish\AppData\Local\Temp\ipykernel_10324\2688343157.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_ckpt = torch.load(checkpoint_inprogress_path, map_locat

Saved: E:\programming\CliniqAI\models\specialists\breast_histology_specialist_v2.pt
Saved: E:\programming\CliniqAI\backend\models\specialists\breast_histology_specialist_v2.pt
Saved: E:\programming\CliniqAI\models\specialists\breast_histology_specialist_v2.history.json


## Step 14 — Test-set evaluation: overall + per-magnification + per-subtype breakdown

In [15]:
model.eval()
all_preds, all_labels, all_mags, all_subtypes = [], [], [], []

with torch.no_grad():
    for (images, labels), recs in zip(test_loader, [test_records[i:i+BATCH_SIZE] for i in range(0, len(test_records), BATCH_SIZE)]):
        images = images.to(device, memory_format=torch.channels_last)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_mags.extend(r[3] for r in recs)
        all_subtypes.extend(r[2] for r in recs)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
overall_acc = (all_preds == all_labels).mean()
print(f"Overall test accuracy: {overall_acc:.4f} (n={len(all_labels)})")

print("\nPer-magnification accuracy:")
for mag in sorted(set(all_mags)):
    idx = [i for i, m in enumerate(all_mags) if m == mag]
    acc = (all_preds[idx] == all_labels[idx]).mean()
    print(f"  {mag}: {acc:.4f} (n={len(idx)})")

print("\nPer-subtype accuracy:")
for subtype in sorted(set(all_subtypes)):
    idx = [i for i, s in enumerate(all_subtypes) if s == subtype]
    if not idx:
        continue
    acc = (all_preds[idx] == all_labels[idx]).mean()
    print(f"  {subtype}: {acc:.4f} (n={len(idx)})")


Overall test accuracy: 0.9162 (n=943)

Per-magnification accuracy:
  100X: 0.9313 (n=262)
  200X: 0.9414 (n=239)
  400X: 0.8852 (n=209)
  40X: 0.9013 (n=233)

Per-subtype accuracy:
  adenosis: 0.9669 (n=121)
  ductal_carcinoma: 0.9391 (n=345)
  fibroadenoma: 0.9894 (n=94)
  lobular_carcinoma: 1.0000 (n=103)
  mucinous_carcinoma: 0.8222 (n=90)
  papillary_carcinoma: 0.4769 (n=65)
  phyllodes_tumor: 1.0000 (n=60)
  tubular_adenoma: 0.9538 (n=65)


## Step 15 — Real-world sanity check

Loads whatever's in `data/raw/manual_sanity_check/` (the same folder used for gatekeeper sanity
testing) and runs this specialist on any breast-histology-looking images there — since in-
distribution test accuracy alone has already been shown (on the gatekeeper) to not guarantee
real-world generalization, don't skip this step.

In [16]:
sanity_images = list(SANITY_DIR.glob("*.png")) + list(SANITY_DIR.glob("*.jpg")) + list(SANITY_DIR.glob("*.jpeg"))
print(f"Found {len(sanity_images)} sanity images in {SANITY_DIR}")

model.eval()
for img_path in sanity_images:
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"{img_path.name}: failed to open ({e})")
        continue
    x = eval_transform(img).unsqueeze(0).to(device, memory_format=torch.channels_last)
    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            probs = torch.softmax(model(x), dim=1)[0]
    pred_idx = probs.argmax().item()
    print(f"\n{img_path.name}:")
    for i, c in enumerate(class_names):
        marker = " <-- predicted" if i == pred_idx else ""
        print(f"  {c}: {probs[i].item():.4f}{marker}")


Found 1 sanity images in E:\programming\CliniqAI\data\raw\manual_sanity_check

Normal_breast_histology.jpg:
  benign: 0.9361 <-- predicted
  malignant: 0.0639


## Notes

- Dataset: BreakHis (Spanhol et al., IEEE TBME 2016), via Kaggle mirror `ambarish/breakhis`.
  Free for non-commercial research/educational use — cite the original paper if this goes in the
  report. 7,909 images, 82 patients, 700x460px RGB, 40X/100X/200X/400X magnifications.
- Split is **patient-grouped** (not image-grouped) to prevent same-patient tissue leaking across
  train/val/test — BreakHis has multiple images per patient across magnifications.
- Class semantics changed from the old `idc_negative`/`idc_positive` to `benign`/`malignant`
  (or 8-way subtype if `MULTICLASS_SUBTYPES=True`) — update `backend/app/schemas.py` and any
  frontend label mapping before deploying this checkpoint.
- `backend/app/config.py`'s specialist registry entry for `breast_histology` needs repointing to
  `breast_histology_specialist_v2.pt` once this is validated — do this only after the sanity
  check above looks right, same discipline as the gatekeeper v3 checkpoints not being deployed
  until they pass their own sanity check.
- Worth committing `scripts/breast_histology_dataset_v2.py` and this notebook to a new branch
  (e.g. `feature/breast-histology-v2`) once training succeeds.
- Next candidate step (separate task, not run here): feed this same BreakHis data into the
  Stage 1 gatekeeper's breast_histology positive class, since its natural magnification range
  may resolve the zoom/crop-scale hypothesis better than the resolution-degradation augmentation
  alone did.